In [1]:
# Evaluation harness: run the golden set through the FULL pipeline and
# measure intent accuracy, execution rate, retry rate, table recall and
# result match - the same metrics Uber's QueryGPT used.
import json
import re
import statistics
from datetime import datetime, timezone
from pathlib import Path

print(f"eval started: {datetime.now(timezone.utc).isoformat()}")

eval started: 2026-11-23T08:03:41.701384+00:00


In [2]:
# The agent stack lives in src/ - every function is defined first in
# notebooks 03/04/05/06 and published there by their export cells. This
# notebook only loads the catalog + rules and imports; no agent code here.
import sys

import yaml
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

catalog = yaml.safe_load((PROJECT_ROOT / "catalog" / "catalog.yaml").read_text(encoding="utf-8"))
rules = yaml.safe_load((PROJECT_ROOT / "catalog" / "rules.yaml").read_text(encoding="utf-8"))

from src.intent import intent_agent
from src.writer import sql_writer
from src.validator import execute_readonly, validator_agent

print(f"agent stack imported from src/ | catalog v{catalog['catalog_version']} | rules v{rules['rules_version']}")

agent stack imported from src/ | catalog v1.2 | rules v1.2


In [3]:
# Golden set: analyst questions with the known-correct answer.
# - expected_workspaces: what intent_agent should return (None = the question
#   is outside the catalog and must be refused)
# - expected_tables: objects the final SQL should reference. Unqualified for
#   single-workspace questions, qualified SCHEMA.OBJECT for two-workspace ones.
#   Kept minimal (the tables truly needed), so views count as a pass when the
#   pipeline answers through them.
# - golden_sql: optional known-correct query. When present, the harness also
#   compares the pipeline's result rows against the golden rows (strict:
#   same rows, any order). A mismatch is not necessarily wrong - it also
#   flags questions where two valid SQL forms return different data.
GOLDEN_SET = [
    {
        "question": "How many employees are in each department?",
        "expected_workspaces": ["HR"],
        "expected_tables": ["DEPARTMENTS", "EMPLOYEES"],
        "golden_sql": """
        SELECT d.department_name, COUNT(e.employee_id) AS employee_count
        FROM departments d
        LEFT JOIN employees e ON e.department_id = d.department_id
        GROUP BY d.department_name
        ORDER BY d.department_name
        """,
    },
    {
        "question": "Average salary by job title",
        "expected_workspaces": ["HR"],
        "expected_tables": ["EMPLOYEES", "JOBS"],
    },
    {
        "question": "Which stores generated the most revenue last year?",
        "expected_workspaces": ["CO"],
        "expected_tables": ["ORDERS", "ORDER_ITEMS", "STORES"],
    },
    {
        "question": "Which products sold the most units?",
        "expected_workspaces": ["SH"],
        "expected_tables": ["SALES", "PRODUCTS"],
        "golden_sql": """
        SELECT p.prod_name, SUM(s.quantity_sold) AS total_quantity
        FROM sales s
        JOIN products p ON p.prod_id = s.prod_id
        GROUP BY p.prod_name
        ORDER BY total_quantity DESC
        FETCH FIRST 5 ROWS ONLY
        """,
    },
    {
        "question": "Which stores are low on stock for their products?",
        "expected_workspaces": ["CO"],
        "expected_tables": ["INVENTORY", "STORES", "PRODUCTS"],
    },
    {
        "question": "Which customers are in which countries?",
        "expected_workspaces": ["SH"],
        "expected_tables": ["CUSTOMERS", "COUNTRIES"],
    },
    {
        "question": "Show customer names and their cities from the sales data",
        "expected_workspaces": ["SH"],
        "expected_tables": ["SALES", "CUSTOMERS"],
    },
    {
        # policy note: cross-schema matching is currently DISABLED
        # (catalog policy: cross_schema_joins disallowed) - the honest
        # outcome for this question is a refusal, not a cross-schema query
        "question": "Compare the countries listed in the HR system with the countries where we have sales",
        "expected_workspaces": None,
        "expected_tables": [],
    },
    {
        "question": "How much revenue did we make per month?",
        "expected_workspaces": ["SH"],
        "expected_tables": ["SALES", "TIMES"],
    },
    {
        "question": "How many servers do we have in the data center?",
        "expected_workspaces": None,
        "expected_tables": [],
    },
]
print(f"golden set: {len(GOLDEN_SET)} questions "
      f"({sum(1 for g in GOLDEN_SET if g.get('golden_sql'))} with golden SQL)")

golden set: 10 questions (2 with golden SQL)


In [4]:
# Metric helpers, shared by the evaluation loop below
def tables_in_sql(sql, expected_tables):
    # how many of the expected tables appear in the final SQL (word boundary,
    # case-insensitive) - views answering instead of base tables count as found
    found = 0
    for table in expected_tables:
        if re.search(rf"\b{re.escape(table)}\b", sql, re.IGNORECASE):
            found += 1
    return found

def rows_signature(rows, cap=20):
    # order-independent, capped signature of a result set for comparison
    return sorted(tuple(map(str, row)) for row in rows[:cap])

In [5]:
# Run every golden question through the full pipeline:
# intent -> writer -> validator. Golden SQL (when present) is executed with
# the same read-only guard so the comparison is apples to apples.
eval_results = []
for entry in GOLDEN_SET:
    question = entry["question"]
    print(f"\n=== {question}")

    # 1. intent: compare the routed workspace list with the expected one
    workspaces = intent_agent(question, catalog)
    intent_ok = workspaces == entry["expected_workspaces"]
    print(f"    intent: {workspaces} (expected {entry['expected_workspaces']}) "
          f"{'OK' if intent_ok else 'MISMATCH'}")

    record = {
        "question": question,
        "expected_workspaces": entry["expected_workspaces"],
        "routed_workspaces": workspaces,
        "intent_ok": intent_ok,
        "status": "no_route",
        "attempts": 0,
        "tables_found": 0,
        "tables_expected": len(entry["expected_tables"]),
        "result_match": None,   # None = no golden SQL for this entry
    }

    # 2. outside the catalog: the pipeline must refuse, nothing else to check
    if workspaces is None:
        eval_results.append(record)
        continue

    # 3. writer -> validator (repair loop included). One flaky LLM call
    # must not kill the whole evaluation: isolate errors per question.
    try:
        draft = sql_writer(question, workspaces, catalog, rules)
        outcome = validator_agent(question, draft, workspaces, catalog, rules, max_retries=3)
    except Exception as exc:
        record["status"] = "pipeline_error"
        record["error"] = str(exc)[:160]
        print(f"    pipeline error: {str(exc)[:80]}")
        eval_results.append(record)
        continue
    record["status"] = outcome["status"]
    record["attempts"] = outcome.get("attempts", len(outcome["history"]) + 1)
    print(f"    validator: {record['status']} after {record['attempts']} attempt(s)")

    if outcome["status"] == "ok":
        record["tables_found"] = tables_in_sql(outcome["sql"], entry["expected_tables"])

        # 4. result match against the golden query (when one exists)
        if entry.get("golden_sql"):
            try:
                golden_cols, golden_rows = execute_readonly(
                    entry["golden_sql"], workspaces, catalog
                )
                record["result_match"] = (
                    rows_signature(golden_rows) == rows_signature(outcome["rows"])
                )
                print(f"    result match vs golden SQL: {record['result_match']}")
            except Exception as exc:
                # a golden query that fails to run is a golden set bug - say so
                record["result_match"] = None
                print(f"    golden SQL failed to execute: {str(exc)[:80]}")

    eval_results.append(record)


=== How many employees are in each department?


    intent: ['HR'] (expected ['HR']) OK


    validator: ok after 1 attempt(s)
    result match vs golden SQL: False

=== Average salary by job title


    intent: ['HR'] (expected ['HR']) OK


    validator: ok after 1 attempt(s)

=== Which stores generated the most revenue last year?


    intent: ['CO'] (expected ['CO']) OK


... (12 outputs left out)

    intent: None (expected None) OK

=== How much revenue did we make per month?


    intent: ['SH'] (expected ['SH']) OK


    validator: ok after 1 attempt(s)

=== How many servers do we have in the data center?


    intent: None (expected None) OK


In [6]:
# Aggregate metrics over the whole golden set
routed = [r for r in eval_results if r["expected_workspaces"] is not None]
routed_ok = [r for r in routed if r["status"] == "ok"]

intent_accuracy = sum(r["intent_ok"] for r in eval_results) / len(eval_results)
exec_rate = sum(r["status"] == "ok" for r in routed) / len(routed) if routed else 0.0
avg_attempts = statistics.mean(r["attempts"] for r in routed_ok) if routed_ok else 0.0
tables_expected = sum(r["tables_expected"] for r in eval_results)
tables_found = sum(r["tables_found"] for r in eval_results)
table_recall = tables_found / tables_expected if tables_expected else 1.0

golden = [r for r in eval_results if r["result_match"] is not None]
result_match_rate = (
    sum(r["result_match"] for r in golden) / len(golden) if golden else 1.0
)

print(f"intent accuracy  : {intent_accuracy:.0%} ({sum(r['intent_ok'] for r in eval_results)}/{len(eval_results)})")
print(f"execution rate   : {exec_rate:.0%} ({len(routed_ok)}/{len(routed)} routed questions)")
print(f"retry rate       : {avg_attempts:.2f} avg attempts (1.0 = never repaired)")
print(f"table recall     : {table_recall:.0%} ({tables_found}/{tables_expected} expected tables referenced)")
print(f"result match     : {result_match_rate:.0%} ({sum(1 for r in golden if r['result_match'])}/{len(golden)} golden queries)")

intent accuracy  : 100% (10/10)
execution rate   : 88% (7/8 routed questions)
retry rate       : 1.14 avg attempts (1.0 = never repaired)
table recall     : 78% (14/18 expected tables referenced)
result match     : 0% (0/2 golden queries)


In [7]:
# Per-question detail: keep this notebook as the regression check after
# every change to a prompt, the catalog or the rules file
print(f"{'intent':<7} {'exec':<6} {'try':<4} {'recall':<8} {'match':<6} question")
for r in eval_results:
    intent = "OK" if r["intent_ok"] else "FAIL"
    exec_col = r["status"][:5] if r["status"] != "ok" else "ok"
    recall = f"{r['tables_found']}/{r['tables_expected']}"
    match = "-" if r["result_match"] is None else str(r["result_match"])
    print(f"{intent:<7} {exec_col:<6} {r['attempts']:<4} {recall:<8} {match:<6} {r['question']}")
print(f"\nfinished: {datetime.now(timezone.utc).isoformat()}")

intent  exec   try  recall   match  question
OK      ok     1    2/2      False  How many employees are in each department?
OK      ok     1    2/2      -      Average salary by job title
OK      ok     1    3/3      -      Which stores generated the most revenue last year?
OK      ok     1    2/2      False  Which products sold the most units?
OK      faile  4    0/3      -      Which stores are low on stock for their products?
OK      ok     1    2/2      -      Which customers are in which countries?
OK      ok     2    2/2      -      Show customer names and their cities from the sales data
OK      no_ro  0    0/0      -      Compare the countries listed in the HR system with the countries where we have sales
OK      ok     1    1/2      -      How much revenue did we make per month?
OK      no_ro  0    0/0      -      How many servers do we have in the data center?

finished: 2026-11-23T08:04:59.241470+00:00
